# Solver MTVRP con GRASP

## ¿Qué problema resuelve?

Este notebook implementa un solver para el **Multiple Traveling Vehicle Routing Problem (MTVRP)**, también conocido como Problema de Ruteo de Vehículos con Múltiples Vehículos.

**Contexto:** Una empresa tiene un depósito central y varios clientes distribuidos en un mapa. Dispone de una flota de vehículos con capacidad limitada. El objetivo es diseñar las rutas de entrega de forma que **todos los clientes sean atendidos lo antes posible** (minimizando la latencia total), respetando las restricciones de capacidad de cada vehículo.

## Metodología: GRASP

Se utiliza el algoritmo **GRASP** (*Greedy Randomized Adaptive Search Procedure*), una metaheurística que combina:

- **Greedy (Codicioso):** Preferir las opciones más cercanas/baratas en cada paso.
- **Randomized (Aleatorizado):** Introducir aleatoriedad para explorar distintas soluciones.
- **Local Search (Búsqueda Local):** Mejorar cada solución construida mediante intercambios.

El ciclo se repite muchas veces y al final se reporta **la mejor solución encontrada**.

## Estructura del notebook

| Sección | Descripción |
|---|---|
| Imports | Librerías necesarias |
| `calculate_distance` | Distancia euclidiana entre dos puntos |
| `parse_instance` | Lectura y decodificación de archivos de instancia |
| `calculate_latency` | Función objetivo: latencia total de una solución |
| `grasp_constructive` | Fase 1: construcción aleatorizada-golosa |
| `local_search` | Fase 2: mejora por intercambio de clientes dentro de cada ruta |
| `route_duration` | Duración total de una ruta (suma de arcos consecutivos) |
| `reorder_routes` | Paso 2.5: ordena rutas por ratio duración/clientes para reducir latencia |
| `solve_mtvrp_grasp` | Ciclo GRASP completo |
| `main` | Ejecución sobre todas las instancias |

In [ ]:
import os      # Para acceder al sistema de archivos (leer carpetas y rutas)
import re      # Para buscar patrones de texto dentro de los archivos de instancia (Usando expresiones regulares)
import time    # Para medir cuánto tarda en ejecutarse el algoritmo
import math    # Para calcular raíces cuadradas (necesarias en la distancia euclidiana)
import random  # Para elegir nodos al azar dentro de la lista de candidatos (parte clave del GRASP)

## Función Auxiliar: Distancia Euclidiana (`calculate_distance`)

Calcula la distancia en línea recta entre dos puntos en el plano usando el **Teorema de Pitágoras**:

![Formula-distancia-euclidiana](./imgs/images.png)

Esta función se usa al construir la matriz de distancias cuando la instancia proporciona coordenadas geográficas en lugar de una matriz de tiempos directa.

In [ ]:
def calculate_distance(node1, node2):
    """
    Calcula la distancia euclidiana (en línea recta) entre dos nodos.

    Los nodos se representan como tuplas (x, y) con sus coordenadas en el plano.
    La fórmula usada es: sqrt((x2 - x1)^2 + (y2 - y1)^2)

    Parámetros:
        node1 (tuple): Coordenadas (x, y) del primer nodo.
        node2 (tuple): Coordenadas (x, y) del segundo nodo.

    Retorna:
        float: Distancia en línea recta entre los dos nodos.
    """
    # Diferencia en el eje X elevada al cuadrado
    dx_squared = (node1[0] - node2[0]) ** 2

    # Diferencia en el eje Y elevada al cuadrada
    dy_squared = (node1[1] - node2[1]) ** 2

    # Raíz cuadrada de la suma: distancia euclidiana
    return math.sqrt(dx_squared + dy_squared)

## Lectura de Instancias (`parse_instance`)

Esta función lee un archivo de texto con los datos del problema y los convierte en estructuras de Python que el algoritmo puede usar.

Los archivos de instancia pueden tener dos formatos:

| Formato | Descripción |
|---|---|
| `CoorX / CoorY` | El archivo contiene coordenadas (x, y) de cada nodo. Las distancias se calculan con la fórmula euclidiana. |
| `TravelTimes` | El archivo contiene directamente una matriz con los tiempos de viaje entre cada par de nodos. |

**Datos extraídos:**
- **`nodes`**: posición de cada nodo (depósito = nodo 0, clientes = nodos 1, 2, ...).
- **`demands`**: cuánto carga necesita cada cliente.
- **`capacity`**: cuánta carga puede llevar cada vehículo.
- **`dist_matrix`**: tabla de distancias/tiempos entre todos los pares de nodos.

In [ ]:
def parse_instance(filepath):
    """
    Lee y decodifica un archivo de instancia del problema MTVRP.

    Soporta dos formatos:
      - Coordenadas (CoorX / CoorY): calcula las distancias con la fórmula euclidiana.
      - Matriz de tiempos (TravelTimes): usa los tiempos ya dados en el archivo.

    Parámetros:
        filepath (str): Ruta al archivo .txt con la instancia del problema.

    Retorna:
        nodes       (dict): Diccionario {id_nodo: (x, y)}. El nodo 0 es el depósito.
        demands     (dict): Diccionario {id_nodo: demanda}. Cuánto pide cada cliente.
        capacity    (float): Capacidad máxima de carga de cada vehículo.
        max_time    (float): Tiempo máximo por ruta (se usa inf si no está definido).
        dist_matrix (dict): Diccionario anidado {i: {j: distancia}} con todas las distancias.
    """
    # Leer todo el archivo de texto de una sola vez
    with open(filepath, 'r') as f:
        content = f.read()

    # Buscar la capacidad del vehículo usando expresión regular
    # re.search devuelve la primera coincidencia del patrón en el texto
    match_cap = re.search(r'VehCapacity:?\s*(\d+)', content)
    capacity = float(match_cap.group(1)) if match_cap else 0  # Si no se encuentra, se asume 0

    # El nodo 0 es el depósito (punto de salida y llegada), su demanda es 0
    demands = {0: 0.0}

    # Extraer las demandas de cada cliente del bloque "ClientDemands"
    match_demands = re.search(r'ClientDemands:\s*\n([0-9\s]+)\nServiceTimes:', content)
    if match_demands:
        # Convertir la cadena de números a una lista de flotantes
        demands_list = [float(x) for x in match_demands.group(1).split()]
        # Asignar cada demanda a su cliente (los clientes empiezan en el nodo 1)
        for i, d in enumerate(demands_list):
            demands[i + 1] = d  # i+1 porque el nodo 0 es el depósito

    nodes = {}        # Diccionario de posiciones de cada nodo
    dist_matrix = {}  # Matriz de distancias entre todos los pares de nodos

    if 'CoorX' in content and 'CoorY' in content:
        # --- Formato con coordenadas geográficas ---
        # Extraer todo lo que viene después de la etiqueta "CoorY"
        coor_str = content.split('CoorY')[1].strip()
        lines = [line.strip() for line in coor_str.split('\n') if line.strip()]

        # Cada línea tiene dos valores: coordenada X y coordenada Y
        for i, line in enumerate(lines):
            parts = line.split()
            nodes[i] = (float(parts[0]), float(parts[1]))  # Tupla (x, y)

        # Calcular la distancia entre cada par de nodos (i, j) con fórmula euclidiana
        for i in nodes:
            dist_matrix[i] = {}
            for j in nodes:
                dist_matrix[i][j] = calculate_distance(nodes[i], nodes[j])

    elif 'TravelTimes:' in content:
        # --- Formato con matriz de tiempos precomputada ---
        # Extraer el bloque numérico que viene después de "TravelTimes:"
        matrix_str = content.split('TravelTimes:')[1].strip()
        lines = [line.strip() for line in matrix_str.split('\n') if line.strip()]

        # Cada fila de la matriz corresponde al nodo i; cada columna, al nodo j
        for i, line in enumerate(lines):
            parts = line.split()
            dist_matrix[i] = {}
            for j, val in enumerate(parts):
                dist_matrix[i][j] = float(val)

        # En este formato no hay coordenadas reales; asignamos (0,0) como marcador
        nodes = {i: (0, 0) for i in range(len(lines))}

    # Se retorna inf como tiempo máximo porque este dato no suele estar en las instancias
    return nodes, demands, capacity, float('inf'), dist_matrix

## Función Objetivo: Latencia Total (`calculate_latency`)

El objetivo del MTVRP en este notebook es **minimizar la latencia total**, que se define como:

\[
\text{Latencia total} = \sum_{\text{cada cliente}} \text{tiempo de llegada a ese cliente}
\]

A diferencia de minimizar la distancia total recorrida, minimizar la latencia premia **atender a los clientes lo antes posible**. Un cliente que espera mucho tiempo contribuye más a la latencia que uno atendido rápidamente.

**Importante:** el reloj (`global_time`) avanza de forma continua a lo largo de todas las rutas, no se reinicia entre vehículos. Esto refleja que todos los clientes perciben el tiempo desde el mismo instante inicial (el comienzo de la operación).

In [ ]:
def calculate_latency(routes, dist_matrix):
    """
    Calcula la latencia total de una solución: la suma de los tiempos de llegada
    a cada cliente considerando el orden global en que son atendidos.

    La latencia mide cuánto tiempo acumula cada cliente esperando ser visitado
    desde el inicio de la operación. Un cliente visitado tarde contribuye más
    a la latencia total que uno visitado temprano.

    Parámetros:
        routes      (list): Lista de rutas. Cada ruta es una lista de nodos
                            que empieza y termina en el depósito (nodo 0).
                            Ejemplo: [[0, 3, 1, 0], [0, 2, 4, 0]]
        dist_matrix (dict): Diccionario anidado con las distancias entre nodos.

    Retorna:
        float: Suma total de los tiempos de llegada a todos los clientes.
    """
    total_latency = 0  # Acumulador de la latencia total de la solución
    global_time = 0    # Reloj global que avanza conforme los vehículos se mueven

    for route in routes:
        # Recorrer cada par consecutivo de nodos dentro de la ruta
        for i in range(len(route) - 1):
            curr_node = route[i]       # Nodo actual (desde donde se viaja)
            next_node = route[i + 1]   # Nodo siguiente (hacia donde se viaja)

            # Tiempo que tarda en ir del nodo actual al siguiente
            dist = dist_matrix[curr_node][next_node]

            # El reloj global avanza ese tiempo
            global_time += dist

            # Solo contamos la latencia al llegar a clientes, no al regresar al depósito
            if next_node != 0:
                total_latency += global_time  # El cliente espera desde el inicio hasta que llega el vehículo

    return total_latency

## Fase 1: Construcción Aleatorizada Greedy (`grasp_constructive`)

Esta función implementa el corazón del GRASP: construir rutas de forma **semialeatoria**.

**¿Por qué no elegir siempre el cliente más cercano (greedy puro)?**  
Porque siempre elegir el más cercano puede llevar a soluciones que son buenas localmente pero malas globalmente. Al introducir aleatoriedad, cada ejecución explora una solución diferente.

**Mecanismo: Lista Restringida de Candidatos (RCL)**  
En lugar de elegir al cliente más cercano directamente, se forma una lista con los `alpha` clientes más cercanos que respetan las restricciones de capacidad y tiempo. Luego se elige **uno al azar** de esa lista.

**Restricciones verificadas en cada paso:**
- **Capacidad**: la demanda del nuevo cliente no debe exceder la carga disponible del vehículo.
- **Tiempo**: debe haber tiempo suficiente para visitar al cliente *y* regresar al depósito.

Si ningún cliente puede ser atendido por el vehículo actual, este **regresa al depósito** y se asigna un vehículo nuevo.

In [ ]:
def grasp_constructive(nodes, demands, capacity, max_time, dist_matrix, alpha=3):
    """
    Fase 1 del GRASP: Construcción Aleatorizada Golosa (Greedy Randomized).

    Construye rutas asignando clientes a vehículos de forma iterativa.
    En cada paso, no elige simplemente al cliente más cercano (puro greedy),
    sino que forma una lista de los 'alpha' mejores candidatos y elige
    uno de ellos al azar. Esto introduce diversidad en cada iteración.

    Parámetros:
        nodes       (dict): Diccionario {id_nodo: (x, y)} con todos los nodos.
        demands     (dict): Diccionario {id_nodo: demanda} de cada cliente.
        capacity    (float): Capacidad máxima de cada vehículo.
        max_time    (float): Tiempo máximo permitido por ruta.
        dist_matrix (dict): Matriz de distancias entre todos los nodos.
        alpha       (int):  Tamaño de la Lista Restringida de Candidatos (RCL).
                            Un alpha mayor da más aleatoriedad; uno menor, más greedy.

    Retorna:
        list: Lista de rutas, donde cada ruta es una lista de nodos
              que comienza y termina en el depósito (nodo 0).
    """
    # Conjunto de clientes aún no asignados a ninguna ruta (se excluye el depósito, nodo 0)
    unvisited = set(nodes.keys())
    unvisited.remove(0)  # El depósito no necesita ser "visitado" como cliente

    routes = []          # Lista final con todas las rutas construidas
    current_route = [0]  # La ruta actual empieza siempre en el depósito
    current_load = 0     # Carga acumulada en el vehículo de la ruta actual
    current_route_time = 0  # Tiempo transcurrido en la ruta actual
    current_node = 0     # Nodo donde se encuentra actualmente el vehículo (comienza en depósito)

    # Continuar hasta que todos los clientes hayan sido asignados
    while unvisited:
        candidates = []  # Lista de clientes que SÍ pueden ser visitados desde el nodo actual

        for candidate in unvisited:
            dist = dist_matrix[current_node][candidate]  # Distancia al cliente candidato

            # Verificar si agregar este cliente excedería la capacidad del vehículo
            demand_fits = (current_load + demands[candidate]) <= capacity

            # Verificar si hay tiempo suficiente para visitar al cliente y regresar al depósito
            time_to_return = dist_matrix[candidate][0]  # Tiempo de regreso al depósito desde el candidato
            time_fits = (current_route_time + dist + time_to_return) <= max_time

            # Solo se agrega a candidatos si cumple ambas restricciones
            if demand_fits and time_fits:
                candidates.append((dist, candidate))  # Guardamos (distancia, id_cliente)

        if candidates:
            # --- Lógica RCL (Restricted Candidate List) ---
            # Ordenar candidatos por distancia (el más cercano primero)
            candidates.sort(key=lambda x: x[0])

            # Tomar solo los 'alpha' mejores (más cercanos)
            rcl = candidates[:alpha]

            # Elegir UNO al azar de esa lista reducida (aquí entra la aleatoriedad del GRASP)
            chosen = random.choice(rcl)
            best_dist, best_next_node = chosen

            # Agregar el cliente elegido a la ruta actual y actualizar el estado
            current_route.append(best_next_node)
            unvisited.remove(best_next_node)   # Ya no está pendiente
            current_load += demands[best_next_node]  # Aumenta la carga del vehículo
            current_route_time += best_dist    # Avanza el tiempo de la ruta
            current_node = best_next_node      # El vehículo ahora está en este nodo
        else:
            # No hay ningún cliente que quepa en el vehículo actual:
            # cerrar la ruta regresando al depósito y abrir una nueva ruta
            current_route.append(0)        # El vehículo regresa al depósito
            routes.append(current_route)   # Guardar la ruta completada

            # Reiniciar el estado para el nuevo vehículo
            current_route = [0]
            current_load = 0
            current_route_time = 0
            current_node = 0

    # Si la última ruta no regresó al depósito, cerrarla manualmente
    if current_route[-1] != 0:
        current_route.append(0)
        routes.append(current_route)

    return routes

## Fase 2: Búsqueda Local (`local_search`)

Una vez que la fase constructiva genera una solución inicial, la **búsqueda local** intenta mejorarla.

**Estrategia: intercambio de pares (swap)**  
Se prueban todos los pares de posiciones \((i, j)\) dentro de cada ruta. Si intercambiar los clientes en esas posiciones reduce la latencia total, el intercambio se acepta y el proceso comienza de nuevo desde la primera ruta.

El algoritmo termina cuando **ningún intercambio** produce una mejora: se dice que la solución es un **óptimo local**.

> Esta búsqueda es exhaustiva dentro de cada ruta pero no cruza clientes entre rutas distintas (intra-ruta).

In [ ]:
def local_search(routes, dist_matrix):
    """
    Fase 2 del GRASP: Búsqueda Local por intercambio (swap) dentro de cada ruta.

    Toma la solución construida y la mejora iterativamente. En cada paso,
    intenta intercambiar el orden de visita de dos clientes dentro de la misma
    ruta. Si el intercambio reduce la latencia total, se acepta y se repite
    el proceso hasta que ningún intercambio mejore la solución.

    Esta técnica se llama "2-opt intra-ruta" o simplemente "swap local".

    Parámetros:
        routes      (list): Lista de rutas actuales (resultado de la fase constructiva).
        dist_matrix (dict): Matriz de distancias entre nodos.

    Retorna:
        best_routes   (list):  Las rutas con la menor latencia encontrada.
        best_latency  (float): El valor de latencia de esas rutas.
    """
    improved = True  # Bandera que controla si se encontró alguna mejora en la última pasada

    # Hacemos copias profundas de las rutas para no modificar las originales
    # r[:] crea una copia de la lista r (cada ruta es una lista de nodos)
    best_routes = [r[:] for r in routes]
    best_latency = calculate_latency(best_routes, dist_matrix)  # Latencia de la solución inicial

    # Continuar buscando mejoras mientras se encuentre al menos una en cada pasada
    while improved:
        improved = False  # Asumir que no habrá mejora; si la hay, se marcará como True

        # Revisar cada ruta individualmente
        for r_idx in range(len(best_routes)):
            route = best_routes[r_idx]

            # Probar intercambiar cada par de clientes (i, j) dentro de esta ruta
            # Los índices empiezan en 1 y terminan en len-2 para evitar tocar el depósito (índice 0 y el último)
            for i in range(1, len(route) - 2):
                for j in range(i + 1, len(route) - 1):
                    # Crear copias de todas las rutas para probar el intercambio sin alterar las originales
                    new_routes = [r[:] for r in best_routes]

                    # Intercambiar (swap) los clientes en las posiciones i y j de esta ruta
                    new_routes[r_idx][i], new_routes[r_idx][j] = new_routes[r_idx][j], new_routes[r_idx][i]

                    # Calcular la latencia con el intercambio aplicado
                    new_latency = calculate_latency(new_routes, dist_matrix)

                    # Si mejoró, actualizar la mejor solución conocida y reiniciar la búsqueda
                    if new_latency < best_latency:
                        best_latency = new_latency
                        best_routes = new_routes
                        improved = True  # Hubo mejora; habrá otra pasada completa
                        break           # Salir del bucle j y reiniciar desde el principio
                if improved:
                    break  # Salir del bucle i
            if improved:
                break  # Salir del bucle de rutas y volver al while

    return best_routes, best_latency

## Paso 2.5: Reordenamiento de Rutas (`reorder_routes`)

Una vez que la búsqueda local ha mejorado cada ruta internamente, queda una optimización adicional que actúa sobre el **orden en que se ejecutan las rutas**.

### ¿Por qué importa el orden?

La función `calculate_latency` usa un **reloj global continuo** que no se reinicia entre rutas. Esto significa que si una ruta lenta se ejecuta primero, todos los clientes de las rutas siguientes acumulan ese tiempo de espera adicional, disparando la latencia total.

### Estrategia: ordenar por ratio duración / clientes

Para cada ruta se calcula el ratio:

\[
\text{ratio}_k = \frac{\text{duración de la ruta } k}{\text{número de clientes en la ruta } k}
\]

Las rutas se ordenan de **menor a mayor ratio**: las más compactas (que sirven a más clientes en menos tiempo) se ejecutan primero. Esto garantiza que el mayor número posible de clientes sea atendido con el reloj global en su valor más bajo.

### Funciones involucradas

| Función | Rol |
|---|---|
| `route_duration` | Calcula la distancia total recorrida en una sola ruta sumando los arcos consecutivos. |
| `reorder_routes` | Ordena la lista de rutas usando `route_duration` como base del ratio, descartando rutas vacías. |

> Este reordenamiento es un **post-procesamiento de costo casi nulo** (solo requiere ordenar una lista pequeña) pero puede reducir la latencia de forma significativa en instancias con rutas de duraciones muy distintas.

In [ ]:
def route_duration(route, dist_matrix):
    """
    Calcula la duración total de una ruta sumando las distancias entre nodos consecutivos.

    Recorre todos los arcos de la ruta y acumula la distancia recorrida desde el
    primer nodo hasta el último (normalmente del depósito al depósito).

    Parámetros:
        route       (list): Lista de nodos que forman la ruta, incluyendo el depósito
                            al inicio y al final. Ejemplo: [0, 3, 1, 5, 0].
        dist_matrix (dict): Diccionario anidado {i: {j: distancia}} entre nodos.

    Retorna:
        float: Distancia total recorrida a lo largo de la ruta.
    """
    d = 0
    for i in range(len(route) - 1):
        d += dist_matrix[route[i]][route[i + 1]]
    return d


def reorder_routes(routes, dist_matrix):
    """
    Reordena las rutas para reducir la latencia total de la solución.

    Dado que calculate_latency emplea un reloj global continuo (no se reinicia entre
    rutas), el orden en que se ejecutan las rutas afecta directamente la latencia:
    ejecutar primero las rutas más compactas (menor tiempo por cliente) reduce el
    tiempo de espera acumulado de todos los clientes.

    Métrica de ordenamiento:
        ratio = duración_de_ruta / número_de_clientes

    Las rutas con menor ratio se colocan primero. Las rutas vacías se descartan.

    Parámetros:
        routes      (list): Lista de rutas a ordenar. Cada ruta es una lista de nodos
                            que comienza y termina en el depósito (nodo 0).
        dist_matrix (dict): Diccionario anidado {i: {j: distancia}} entre nodos.

    Retorna:
        list: Nueva lista de rutas ordenadas de menor a mayor ratio duración/clientes.
    """
    route_info = []
    for r in routes:
        clients_count = len(r) - 2  # Excluir el depósito al inicio y al final
        if clients_count <= 0:
            continue  # Ignorar rutas vacías o inválidas
        dur = route_duration(r, dist_matrix)
        ratio = dur / clients_count
        route_info.append((ratio, r))

    # Ordenar de menor a mayor ratio: las rutas más compactas van primero
    route_info.sort(key=lambda x: x[0])
    return [r for _, r in route_info]

## GRASP Completo (`solve_mtvrp_grasp`)

Esta función **orquesta el ciclo GRASP** completo. Repite `iterations` veces el siguiente ciclo:

```
Para cada iteración:
    1. Construir  → grasp_constructive()  (solución nueva con aleatoriedad)
    2. Mejorar    → local_search()        (swap de clientes dentro de rutas)
    3. Comparar   → si es mejor que la mejor conocida, guardarla
```

Al final retorna **la mejor solución encontrada** entre todas las iteraciones.

> El número de iteraciones es un balance entre **calidad de solución** y **tiempo de cómputo**. Con más iteraciones se exploran más combinaciones posibles.

In [ ]:
def solve_mtvrp_grasp(nodes, demands, capacity, max_time, dist_matrix, iterations=50, alpha=3):
    """
    Orquesta la metodología GRASP completa para resolver el MTVRP.

    GRASP (Greedy Randomized Adaptive Search Procedure) repite dos fases
    muchas veces y se queda con la mejor solución encontrada:
        1. Construcción aleatoria-golosa: genera una solución nueva cada vez.
        2. Búsqueda local: mejora esa solución intercambiando clientes.

    Gracias a la aleatoriedad en la construcción, cada iteración puede
    explorar una región diferente del espacio de soluciones, aumentando
    las probabilidades de encontrar una buena solución global.

    Parámetros:
        nodes       (dict): Diccionario de nodos {id: (x, y)}.
        demands     (dict): Demanda de cada cliente {id: demanda}.
        capacity    (float): Capacidad máxima de cada vehículo.
        max_time    (float): Tiempo máximo por ruta.
        dist_matrix (dict): Matriz de distancias entre nodos.
        iterations  (int):  Número de veces que se repite el ciclo Construir + Mejorar.
                            Más iteraciones = más tiempo, pero posiblemente mejor solución.
        alpha       (int):  Tamaño de la Lista Restringida de Candidatos (RCL) pasado a
                            grasp_constructive. Permite comparar configuraciones distintas.

    Retorna:
        best_overall_routes   (list):  Las rutas con la latencia más baja encontrada.
        best_overall_latency  (float): El valor mínimo de latencia logrado.
    """
    best_overall_routes = None          # Guardará las mejores rutas vistas hasta ahora
    best_overall_latency = float('inf') # Inicializar con "infinito" para que cualquier solución sea mejor

    for _ in range(iterations):
        # --- Fase 1: Construir una solución nueva (con aleatoriedad) ---
        routes = grasp_constructive(nodes, demands, capacity, max_time, dist_matrix, alpha=alpha)

        # --- Fase 2: Mejorar esa solución con búsqueda local ---
        improved_routes, _ = local_search(routes, dist_matrix)

        # --- Reordenar rutas para minimizar latencia secuencial ---
        sorted_routes = reorder_routes(improved_routes, dist_matrix)
        sorted_latency = calculate_latency(sorted_routes, dist_matrix)

        # --- Comparar: si esta iteración dio mejor resultado, guardarlo ---
        if sorted_latency < best_overall_latency:
            best_overall_latency = sorted_latency
            best_overall_routes = sorted_routes

    return best_overall_routes, best_overall_latency

## Función Principal (`main`)

Esta función actúa como el **punto de entrada** del programa.

Su flujo es:
1. Verificar que exista la carpeta `./instancias` con los archivos de prueba.
2. Definir los valores de alpha a explorar: `ALPHA_VALUES = [1, 2, 3, 5, 7]`.
3. Por cada archivo `.txt`, ejecutar el GRASP **una réplica por cada valor de alpha** y recolectar estadísticas.
4. Mostrar los resultados en una tabla con las columnas:

| Columna | Descripción |
|---|---|
| **Instancia** | Nombre del archivo procesado |
| **Mejor Valor Enc.** | Mínimo valor de la función objetivo (latencia) entre todas las réplicas |
| **Valor Promedio** | Promedio de la latencia entre réplicas |
| **Peor Valor** | Máximo valor de latencia entre réplicas |
| **Tiempo Promedio (s)** | Tiempo de ejecución promedio por réplica |
| **Num. de Réplicas** | Total de réplicas ejecutadas (= número de alphas probados) |
| **Mejor Config. (alpha)** | Valor de alpha (tamaño de la RCL) que produjo la mejor solución |

In [ ]:
def main():
    """
    Función principal: lee todas las instancias de la carpeta './instancias'
    y ejecuta el algoritmo GRASP sobre cada una con distintos valores de alpha,
    mostrando estadísticas comparativas en tabla.

    Por cada archivo .txt encontrado:
        1. Se parsea la instancia (nodos, demandas, capacidad, matriz de distancias).
        2. Se ejecuta GRASP con 50 iteraciones para cada valor en ALPHA_VALUES.
        3. Se imprime una fila con: mejor valor encontrado, valor promedio, peor valor,
           tiempo promedio, número de réplicas y la mejor configuración (alpha).

    No recibe parámetros ni retorna valores; su propósito es mostrar resultados
    directamente en pantalla (efecto de lado / side effect).
    """
    folder_path     = "./instancias"
    ALPHA_VALUES    = [1, 2, 3, 5, 7]          # Valores de alpha (tamaño RCL) a comparar
    NUM_REPLICATIONS = len(ALPHA_VALUES)        # Una réplica por cada valor de alpha

    if not os.path.exists(folder_path):
        print(f"Crea la carpeta '{folder_path}' y coloca las instancias ahí.")
        return

    # Anchos de columna para formatear la tabla
    col_inst     = 22
    col_best_val = 22
    col_avg_val  = 16
    col_worst    = 14
    col_time     = 20
    col_reps     = 20
    col_config   = 22

    header = (
        f"{'Instancia':<{col_inst}} | "
        f"{'Mejor Valor Enc.':<{col_best_val}} | "
        f"{'Valor Promedio':<{col_avg_val}} | "
        f"{'Peor Valor':<{col_worst}} | "
        f"{'Tiempo Promedio (s)':<{col_time}} | "
        f"{'Num. de Replicas':<{col_reps}} | "
        f"{'Mejor Config. (alpha)':<{col_config}}"
    )
    print("\n")
    print(header)
    print("-" * len(header))

    all_best_routes = {}

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt") or filename.endswith(".TXT"):
            filepath = os.path.join(folder_path, filename)
            try:
                nodes, demands, capacity, max_time, dist_matrix = parse_instance(filepath)

                latencies    = []
                times        = []
                best_routes  = None
                best_latency = float('inf')
                best_alpha   = None

                # Una réplica por cada valor de alpha
                for alpha in ALPHA_VALUES:
                    t0 = time.time()
                    routes, latency = solve_mtvrp_grasp(
                        nodes, demands, capacity, max_time, dist_matrix,
                        iterations=50, alpha=alpha
                    )
                    times.append(time.time() - t0)
                    latencies.append(latency)
                    if latency < best_latency:
                        best_latency = latency
                        best_routes  = routes
                        best_alpha   = alpha

                best_val  = min(latencies)
                avg_val   = sum(latencies) / len(latencies)
                worst_val = max(latencies)
                avg_time  = sum(times) / len(times)

                print(
                    f"{filename:<{col_inst}} | "
                    f"{best_val:<{col_best_val}.2f} | "
                    f"{avg_val:<{col_avg_val}.2f} | "
                    f"{worst_val:<{col_worst}.2f} | "
                    f"{avg_time:<{col_time}.4f} | "
                    f"{NUM_REPLICATIONS:<{col_reps}} | "
                    f"{best_alpha:<{col_config}}"
                )
                all_best_routes[filename] = best_routes

            except Exception as e:
                print(f"{filename:<{col_inst}} | ERROR AL LEER: {e}")

    print("\n")
    print("=" * len(header))
    print("DETALLE DE RUTAS POR INSTANCIA (MEJOR SOLUCION)")
    print("=" * len(header))
    for filename, routes in all_best_routes.items():
        print(f"\nInstancia: {filename}")
        for idx, route in enumerate(routes):
            route_str = " -> ".join(str(n) for n in route)
            print(f"  Viaje {idx + 1}: {route_str}")

    print("\n")

## Ejecución

Ejecuta la celda siguiente para correr el algoritmo sobre todas las instancias de la carpeta `./instancias`.

> **Nota:** Con 100 iteraciones, el proceso puede tardar varios segundos por instancia dependiendo del tamaño del problema.

In [ ]:
main()